<a href="https://colab.research.google.com/github/vchirrav-eng/sec546_notebooks/blob/main/SEC546_14_AIAgent_JIT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Identity: Individual Security Pillars

# 1. Signed Code
Ensures the agent's logic is verified and hasn't been modified by an attacker.

In [ ]:
def verify_signature(logic):
    print(f"[SEC] Verifying hash for: {logic}")
    return True # Simulate success

verify_signature("delete_bucket()")

[SEC] Verifying hash for: delete_bucket()


True

# 2. HITL (Human-In-The-Loop)
A manual checkpoint for sensitive operations.

In [ ]:
def human_gate(action):
    return input(f"Authorize {action}? (y/n): ") == 'y'

if human_gate("Delete Database"):
    print("Proceeding...")

Authorize Delete Database? (y/n): y
Proceeding...


# 3. JIT & Per-Call Scoped Credentials
Credentials are created 'Just In Time' and restricted only to the specific tool being called.

In [ ]:
import hmac
import hashlib
import time
import base64

class JITCredentialManager:
    def __init__(self, secret_key: str):
        self._secret = secret_key.encode()

    def generate_scoped_token(self, agent_id: str, tool_name: str, ttl_seconds: int = 60):
        """Generates a cryptographically bound, time-limited token for a specific tool."""
        expires_at = int(time.time()) + ttl_seconds
        payload = f"{agent_id}:{tool_name}:{expires_at}"

        signature = hmac.new(
            self._secret,
            payload.encode(),
            hashlib.sha256
        ).digest()

        token = base64.b64encode(payload.encode() + b"." + signature).decode()
        return token, expires_at

# --- Production Simulation ---
manager = JITCredentialManager(secret_key="PROD_MASTER_KEY_001")

# Per-call scoping for the 'S3_READ' operation
token, expiry = manager.generate_scoped_token(agent_id="agent_88", tool_name="S3_READ")

print(f"[PROD JIT] Scoped Token: {token}")
print(f"[PROD JIT] Valid until: {time.ctime(expiry)}")
print(f"[SCOPE] This token is strictly rejected if used for 'S3_DELETE' or by 'agent_89'.")

[PROD JIT] Scoped Token: YWdlbnRfODg6UzNfUkVBRDoxNzg5NDAwOTQxLp6zkjo88apQh/DdWuU4h/xqneWGju7jDqMtBiBHeyWW
[PROD JIT] Valid until: Mon Sep 14 15:49:01 2026
[SCOPE] This token is strictly rejected if used for 'S3_DELETE' or by 'agent_89'.


# 4. Audit Trail
Logging every internal decision and external call for forensic review.

In [ ]:
def audit(log_entry):
    print(f"[AUDIT LOG]: {log_entry}")

audit({"timestamp": "12:00:01", "event": "API_CALL", "status": "200"})

[AUDIT LOG]: {'timestamp': '12:00:01', 'event': 'API_CALL', 'status': '200'}
